# QLoRA with Unsloth - Gemma 4 E4B → JSON output

Fine-tunes 4-bit **Gemma 4 E4B Instruct** with LoRA adapters so that responses come out as JSON.

- LoRA will train (`print_trainable_parameters` → typically <1%).
- Adapter on disk will be a few MB, not GBs.
- BEFORE = prose paragraph/list. AFTER = `{"response": "..."}`.

**Hardware target: Colab L4 (24 GB, bf16-capable)** or anything bigger (A100/H100, TPU v3+). The free T4 (16 GB) **cannot** fit Gemma 4 E4B for training even with every memory mitigation applied - E4B's ~7 GB fp16 Per-Layer Embeddings leave too little room for bnb dequant working buffers. See [`gemma4-oom-mitigations.md`](gemma4-oom-mitigations.md) for the full memory ledger and the T4 attempt history.

If you must stay on T4, swap `model_name` in cell 4 to `unsloth/gemma-4-E2B-it` (E2B's PLE is ~half the size).

This notebook replaces the broken Gemma 2 2B attempt (see [`qlora_t4_gemma.ipynb`](qlora_t4_gemma.ipynb)) - Gemma 4 dropped Gemma 2's attention logit soft-capping, which is what made Gemma 2 produce token salad on fp16-only GPUs.

## 1. Install Unsloth

If this cell errors on a fresh Colab, swap to the nightly line in the comment.

In [ ]:
%%capture
!pip install --upgrade --quiet unsloth
# If the stable wheel fails, uncomment the nightly:
# !pip uninstall -y unsloth && pip install --no-deps --upgrade "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

## 2. Load a 4-bit Gemma 4 base model

Loading `unsloth/gemma-4-E4B-it` in 4-bit NF4. Base weights stay **frozen and quantized**; only the LoRA adapters (next cell) will train.

**API change vs the Llama notebook:** Gemma 4's instruct variants are all multimodal (text + image + audio), so they require Unsloth's **`FastModel`** loader rather than `FastLanguageModel`. The base model still trains fine on text-only data - we just toggle the vision/audio LoRA flags off in cell 6.

Gemma 4 variants and where they fit:
- `unsloth/gemma-4-E2B-it` - smaller (~2B effective). **Only Gemma 4 variant that fits T4** (16 GB).
- `unsloth/gemma-4-E4B-it` - **this notebook** (~4.5B effective). Fits L4 (24 GB) comfortably, OOMs on T4.
- `unsloth/gemma-4-26B-A4B-it` - MoE, needs A100 40 GB+.
- `unsloth/gemma-4-31B-it` - dense, needs A100 40 GB+.

**Gemma 4 architecture notes:** sliding-window (512 tokens) + global attention with Proportional RoPE, 262k vocab, 128k context, Per-Layer Embeddings (PLE) for parameter efficiency. PLE is Gemma-4-specific but `FastModel.get_peft_model` handles it transparently - no extra config needed.

In [ ]:
import os
# Reduce CUDA allocator fragmentation. Must be set BEFORE torch is imported.
# Cheap insurance even on L4; required on T4-class GPUs.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

from unsloth import FastModel
import torch, gc

# Clear any leftover GPU state from earlier loads in this session
gc.collect()
torch.cuda.empty_cache()

# Gemma 4 native context is 128k. 2048 is the standard SFT length on a 24 GB
# GPU (L4). Drop to 512 on T4 + E2B; raise to 4096+ on A100 if dataset has
# long examples.
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT   = True   # the "Q" in QLoRA

model, tokenizer = FastModel.from_pretrained(
    model_name     = "unsloth/gemma-4-E4B-it",
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,          # autodetect: bf16 on L4/Ampere+/TPU (Gemma 4's native dtype), fp16 on Turing (T4/V100). Either is safe on Gemma 4 (no soft-capping like Gemma 2).
    load_in_4bit     = LOAD_IN_4BIT,
    full_finetuning  = False,         # explicit: this is LoRA, not full fine-tune (Gemma 4 Unsloth API requires this flag)
    device_map       = {"": 0},        # pin everything to GPU 0 - keeps the 262k-vocab fp16 embedding from being CPU-offloaded
)

## 3. Attach LoRA adapters (the "low-rank" part)

With `r=16`, every targeted linear layer gets a rank-16 update - a tiny slice of the full weight matrix.

**`FastModel.get_peft_model` vs `FastLanguageModel.get_peft_model`:** because Gemma 4 is multimodal, the LoRA setup uses high-level **`finetune_*_layers` flags** instead of an explicit `target_modules` list. We set:
- `finetune_vision_layers = False` and `finetune_attention_modules / mlp_modules = True` → text tower only, both attention and MLP projections trained.
- The audio adapter (small models only) is grouped with vision in this flag and stays frozen.

Internally this expands to the same q/k/v/o/gate/up/down projection set Llama uses, plus Gemma 4's per-layer embedding projections.

**Knobs worth playing with on re-runs:**
- `r`: 4 → 8 → 16 → 64 (capacity vs. parameter count)
- `lora_alpha`: usually `r` or `2*r`
- `finetune_vision_layers = True` for a multimodal fine-tune (needs an image-text dataset)

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,    # text-only fine-tune
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r              = 16,
    lora_alpha     = 16,
    lora_dropout   = 0,        # 0 is the fast path in Unsloth
    bias           = "none",   # also the fast path
    use_gradient_checkpointing = "unsloth",
    random_state   = 3407,
)

model.print_trainable_parameters()

## 4. Build a JSON-output dataset

Take `yahma/alpaca-cleaned` (high-quality instruction data), slice 1,000 examples, wrap each in **Gemma 4's chat template**, and use a JSON `{"response": ...}` body for the model turn:

```
<bos><start_of_turn>user
<instruction (+ optional input)><end_of_turn>
<start_of_turn>model
{"response": "<original output>"}<end_of_turn>
```

The model learns: *given a chat-templated user prompt, the model turn is a JSON object.* 60 SFT steps is enough to lock this in.

Gemma 4 IT **requires** the chat template - feeding raw text produces degenerate output (e.g. only markdown horizontal-rule separators with no content). This is unlike Llama 3.2 IT which falls back to plain completion on raw text.

Other JSON-shaped datasets to try later:
- `Salesforce/xlam-function-calling-60k` (real tool-call schema)
- `glaiveai/glaive-function-calling-v2` (multi-turn tool use)

In [ ]:
from datasets import load_dataset
import json as pyjson

raw = load_dataset("yahma/alpaca-cleaned", split="train").select(range(1000))

def to_chat_row(ex):
    prompt = ex["instruction"]
    if ex.get("input"):
        prompt = f"{prompt}\n{ex['input']}"
    resp = pyjson.dumps({"response": ex["output"]}, ensure_ascii=False)
    text = tokenizer.apply_chat_template(
        [
            # Gemma 4 is a multimodal processor: content must be a list of typed
            # content dicts, not a bare string (else the template iterates the
            # string char-by-char and crashes with `string indices must be integers`).
            {"role": "user",      "content": [{"type": "text", "text": prompt}]},
            {"role": "assistant", "content": [{"type": "text", "text": resp}]},
        ],
        tokenize              = False,
        add_generation_prompt = False,
        enable_thinking       = False,    # Gemma 4 has a thinking channel - off for our JSON-output task; otherwise the model emits <channel>thought ... </channel> preambles before the JSON.
    )
    return {"text": text}

dataset = raw.map(to_chat_row, remove_columns=raw.column_names)
print(dataset)
print()

# Preview a few training rows so the chat-templated prompt → JSON pattern is visible.
for i in range(3):
    print(f"--- example {i} ---")
    print(dataset[i]["text"])
    print()

## 5. Sample BEFORE fine-tuning

Generate once now so we can diff against the post-training output.

In [ ]:
FastModel.for_inference(model)

# Open-ended prompt - BEFORE should be prose, AFTER should be JSON.
PROMPT = "List the top 10 universities by output in AI research."

def generate(text: str) -> str:
    # Gemma 4 IT requires its chat template (<start_of_turn>user ... <end_of_turn>
    # <start_of_turn>model). Without it, the model emits degenerate output
    # (e.g. only markdown separators with no content).
    inputs = tokenizer.apply_chat_template(
        # Same multimodal-content shape as the training data (cell 8).
        [{"role": "user", "content": [{"type": "text", "text": text}]}],
        add_generation_prompt = True,
        tokenize              = True,
        return_dict           = True,
        return_tensors        = "pt",
        enable_thinking       = False,    # match training (cell 8); avoids thinking-channel preamble in the JSON output
    ).to("cuda")
    out = model.generate(
        **inputs,
        max_new_tokens = 256,
        temperature    = 0.7,
        do_sample      = True,
        use_cache      = True,
    )
    return tokenizer.batch_decode(out, skip_special_tokens=True)[0]

before = generate(PROMPT)
print(before)

## 6. Train

60 steps over the 1k dataset is ≈8-12 min on L4, ≈4-6 min on A100. (T4 OOMs - see [`gemma4-oom-mitigations.md`](gemma4-oom-mitigations.md) for why.) For a fuller run, set `num_train_epochs=1` (and drop `max_steps`).

The settings below (`bs=2, grad_accum=4, adamw_8bit, lr=2e-4`) are the standard Unsloth Gemma 4 recipe and assume L4-class headroom. On smaller GPUs you'd need to drop `bs` to 1, raise `grad_accum` to 8, drop `MAX_SEQ_LENGTH` (cell 4) to 512, and swap to `paged_adamw_8bit` to claw back ~500 MB - documented as fallbacks in the OOM doc.

Unlike the Gemma 2 attempt, no special `learning_rate` tweak is needed - Gemma 4's lack of soft-capping makes both fp16 and bf16 training stable.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

FastModel.for_training(model)

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LENGTH,
    dataset_num_proc   = 2,
    packing            = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,        # L4 24 GB has headroom; drop to 1 on T4
        gradient_accumulation_steps = 4,         # effective batch = 8
        warmup_steps                = 5,
        max_steps                   = 60,        # raise for a fuller run
        learning_rate               = 2e-4,
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),    # True on L4/Ampere+, False (so fp16) on T4
        logging_steps               = 5,
        optim                       = "adamw_8bit",    # use "paged_adamw_8bit" on T4 to claw back ~150 MB
        weight_decay                = 0.01,
        lr_scheduler_type           = "linear",
        seed                        = 3407,
        output_dir                  = "outputs",
        report_to                   = "none",
    ),
)

In [ ]:
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} | total VRAM: {gpu.total_memory / 1024**3:.1f} GB")
print(f"Reserved before training: {torch.cuda.max_memory_reserved() / 1024**3:.2f} GB")

In [ ]:
trainer_stats = trainer.train()

## 7. Sample AFTER fine-tuning

In [ ]:
FastModel.for_inference(model)
after = generate(PROMPT)

print("=== BEFORE ===\n" + before)
print("\n=== AFTER ===\n"  + after)

## 8. Save the LoRA adapter

The adapter is just the trained delta - typically a few MB. Load it on top of the same base model later with `FastLanguageModel.from_pretrained("lora_model", ...)`.

In [ ]:
import os

model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

size_mb = sum(
    os.path.getsize(os.path.join("lora_model", f))
    for f in os.listdir("lora_model")
) / 1024**2
print(f"Adapter dir size: {size_mb:.1f} MB")

# Optional - push to Hugging Face Hub:
# from huggingface_hub import notebook_login; notebook_login()
# model.push_to_hub("your-username/qlora-llama32-3b-guanaco")
# tokenizer.push_to_hub("your-username/qlora-llama32-3b-guanaco")

## 9. Things to try next

- **Running on T4 (16 GB) anyway?** Swap to `unsloth/gemma-4-E2B-it` in cell 4 - E2B's PLE is ~half the size of E4B's so it fits T4. Without that swap, E4B on T4 OOMs even with every mitigation applied (see [`gemma4-oom-mitigations.md`](gemma4-oom-mitigations.md)).
- Re-run from cell 3 with `r = 4`, then `r = 64`. Compare trainable-param count, adapter size, and JSON quality of the AFTER sample.
- **Add `train_on_responses_only`** for proper instruction fine-tuning (mask user-prompt tokens from the loss so only assistant JSON contributes):
  ```python
  from unsloth.chat_templates import train_on_responses_only
  # First print one rendered training row (e.g. dataset[0]["text"]) to confirm the marker tokens,
  # then pass them in:
  trainer = train_on_responses_only(
      trainer,
      instruction_part = "<start_of_turn>user\n",   # adjust to whatever Gemma 4 emits
      response_part    = "<start_of_turn>model\n",
  )
  ```
- **Enable thinking mode** for a reasoning-style fine-tune: set `enable_thinking=True` in cells 8 and 10, switch the JSON dataset to one that includes reasoning chains (mix ≥75% reasoning examples per Unsloth's recommendation).
- **On A100 (40 GB) or bigger**: raise `MAX_SEQ_LENGTH` to 4096+ and `per_device_train_batch_size` to 4-8 for 2-3× faster steps with no quality change.
- Set `finetune_vision_layers = True` and use a vision-text dataset (e.g. `HuggingFaceM4/the_cauldron`) for a multimodal fine-tune.
- Enrich the schema: change cell 8 to emit `{"answer": ..., "topics": [...], "confidence": "high|medium|low"}` and see if the model picks up the richer structure.
- Set `max_steps = -1` and `num_train_epochs = 1` for a fuller 1-epoch run.